# Análise de características usando uma Árvore de Decisão

> **Autores:** Alexandre Maciel e Vinicius de Lima.

Esse notebook calcula o F1-score de classificações feitas usando uma Árvore de Decisão nas bases descritas na seguinte tabela:

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
sorted = pd.read_csv("knn/sorted.csv")
sorted

In [ ]:
par_criterion = ["gini", "entropy", "log_loss"]
par_max_depth = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


def executar_experimentos(X, y):
    resultados_cv = []

    for c in par_criterion:
        for d in par_max_depth:
            clf = DecisionTreeClassifier(criterion=c, max_depth=d, random_state=42)
            scores = cross_val_score(clf, X, y, cv=10, scoring="f1_macro")
            resultados_cv.append(
                {"criterion": c, "max_depth": d, "f1_cv": np.mean(scores)}
            )

    df_resultados = pd.DataFrame(resultados_cv)
    top_10 = df_resultados.sort_values(by="f1_cv", ascending=False).head(10)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    resultados_finais = []
    for index, row in top_10.iterrows():
        clf = DecisionTreeClassifier(
            criterion=row["criterion"], max_depth=int(row["max_depth"]), random_state=42
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        f1_holdout = f1_score(y_test, y_pred, average="macro")

        resultados_finais.append(
            {
                "criterion": row["criterion"],
                "max_depth": int(row["max_depth"]),
                "f1_10fold": row["f1_cv"],
                "f1_70_30": f1_holdout,
            }
        )

    return pd.DataFrame(resultados_finais)


configs = []
resultados = []
for i, dataset in enumerate(sorted["dataset"]):
    df = pd.read_csv(f"bases/{dataset}")
    X = df.drop(columns=["label"])
    y = df["label"]

    print(f"executando experimentos para dataset {dataset}")
    print(f"{i + 1} de {len(sorted['dataset'])}")

    experimento = executar_experimentos(X, y)

    conf = {"dataset": dataset}
    resultado_10_fold = {"dataset": dataset, "split": "10-fold"}
    resultado_70_30 = {"dataset": dataset, "split": "70/30"}
    for i, r in experimento.iterrows():
        idx = f"conf. {i + 1}"
        conf[idx] = f"criterion={r['criterion']}, max_depth={r['max_depth']}"
        resultado_10_fold[idx] = r["f1_10fold"]
        resultado_70_30[idx] = r["f1_70_30"]

    resultados.extend([resultado_10_fold, resultado_70_30])
    configs.append(conf)

configs = pd.DataFrame(configs)
resultados = pd.DataFrame(resultados)

In [ ]:
try:
    os.mkdir("dtree")
except FileExistsError:
    pass

In [ ]:
configs.to_csv("dtree/configs.csv")
resultados.to_csv("dtree/results.csv")

display(configs)
display(resultados)
resultados.describe()

NameError: name 'configs' is not defined

# 